# Hot Day + Warm Night + No-Wind Compound Event

In this notebook, we'll be identifying incidents where extreme heat and warm overnight temperatures in the SMUD service territory co-occur with calm wind conditions at the SMUD Solano Wind Farm, representing days of high energy demand and low renewable generation simultaneously.

### Planning Horizon

Identify and prioritize investment opportunities based on the timing of certain impacts. This
requires translating GWLs into time-based estimates to identify when a compound event may
challenge your resiliency. This event is focused on **high demand and low renewable energy supply**.

| Sub-condition | Indicator | Threshold |
|---|---|---|
| Limited renewable energy supply** | Wind speed at Solano Wind Project | < 5 m/s |
| High cooling demand | Daily max temp in SMUD Service Territory | > 90°F |
| No nighttime reprieve | Daily min temp in SMUD Service Territory | > 65°F |

** =  We are using wind speed as a proxy for renewable generation. This does not represent true renewable wind generation capacity. Future efforts will incorporate the newly developed wind renewable generation data.

**Intended Application:** As a user, I want to **<span style="color:red">quantify how often high-demand, low-generation compound heat events will occur in SMUD territory across 10 GWLs, and translate those warming levels into time-based planning horizons to prioritize grid investment decisions.</span>**

**Runtime:** ~10 min. to run through the whole notebook. Modifications to different queries may increase/decrease the overall runtime.

#### Imports

In [ ]:
import os
import sys

import climakitae as ck
import dask
import geopandas as gpd
import xarray as xr
from climakitae.core.data_export import export
from climakitae.new_core.data_access import DataCatalog
from dask.diagnostics import ProgressBar

sys.path.append(os.path.abspath('.'))
from wg10_helpers import (get_gwl_year_map, plot_gwl_bar_chart,
                                plot_gwl_maps, plot_smud_solano_grid_map,
                                plot_subcondition_frequency_maps)

SOLANO_SHAPEFILE = "data/solano_windfarm_area.zip"

cd = ck.ClimateData(verbosity=-1)

## User Selections

Modify these parameters to change the compound event definition, domain, or data resolution.

In [ ]:
GWLS       = [0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0]
territory  = "SMUD Service Territory"
resolution = "d03"    # 3 km

warming_level_window = 15              # ±window yr around GWL crossing (15 → 30 yr total)
n_years = warming_level_window * 2.0   # = 30.0

# Hot day + warm night: fixed thresholds applied to each grid cell independently
tmax_threshold_f = 90.0   # daily max temp (°F)
tmin_threshold_f = 65.0   # daily min temp (°F)

# No-wind at Solano: daily maximum wind speed (wspd10max) < 5 m/s
# wspd10max is kept in native m/s — no unit conversion applied
wind_ms_threshold = 5.0   # m/s

# Sacramento city center (used as reference marker on maps)
SAC_LON, SAC_LAT = -121.4944, 38.5816

## Retrieve Data

In [ ]:
procs = {
    "warming_level": {
        "warming_levels": GWLS,
        "warming_level_window": warming_level_window,
    },
    "clip": territory,
    "filter_unadjusted_models": "yes",
}

# Retrieving t2max WRF data
procs["convert_units"] = "degF"
variable = "t2max"
t2max_data = (
    cd.catalog("cadcat")
        .activity_id("WRF")
        .institution_id("UCLA")
        .table_id("day")
        .grid_label(resolution)
        .variable(variable)
        .processes(procs)
        .get()
)
print("t2max dims:", dict(t2max_data.dims))

In [ ]:
# Retrieving t2min WRF data
procs["convert_units"] = "degF"
variable = "t2min"
t2min_data = (
    cd.catalog("cadcat")
        .activity_id("WRF")
        .institution_id("UCLA")
        .table_id("day")
        .grid_label(resolution)
        .variable(variable)
        .processes(procs)
        .get()
)
print("t2min dims:", dict(t2min_data.dims))

In [ ]:
# cd.reset() is required here because wind data uses a different spatial clip
# (Solano shapefile vs. SMUD territory) and a separate processor config.
cd.reset()
wind_data = (
    cd.catalog("cadcat")
    .activity_id("WRF").institution_id("UCLA")
    .table_id("day").grid_label(resolution)
    .variable("wspd10max")
    .processes({
        "warming_level": {
            "warming_levels": GWLS, 
            "warming_level_window": warming_level_window
        },
        "clip": SOLANO_SHAPEFILE,
        "filter_unadjusted_models": "yes",
    })
    .get()
)
print("Solano winds dims:", dict(wind_data.dims))

In [ ]:
# Collapse Solano spatial grid to a single daily value per sim/GWL.
# The Solano wind farm is small enough that the spatial mean is a reasonable
# scalar proxy for regional wind availability.
wind_data_spatial_mean = wind_data.mean(dim=["x", "y"])

## Compute Spatial Masks for SMUD and Solano Winds

Creates two boolean masks of valid (non-null) grid cells:
- `smud_mask`: covers the SMUD service territory; used to clip figure output
- `solano_mask`: covers the Solano wind farm grid cells; used in the territory overview map

In [ ]:
# Boolean mask of valid (non-null) SMUD grid cells — used to blank out
# cells outside the territory before saving and plotting.
smud_mask = (~t2max_data['t2max'].isel(sim=0, time_delta=0, warming_level=0).isnull()).compute()
solano_mask = (~wind_data.isel(warming_level=0, sim=0, time_delta=0).wspd10max.isnull()).compute()

## Threshold Diagnostics

Prints how frequently each sub-condition occurs at the lowest GWL as a sanity check.

In [ ]:
with ProgressBar():
    print("Computing baseline frequencies...")
    _tmax_bl, _tmin_bl, _wind_bl = dask.compute(
        t2max_data.sel(warming_level=GWLS[0])["t2max"].median(dim="sim"),
        t2min_data.sel(warming_level=GWLS[0])["t2min"].median(dim="sim"),
        wind_data_spatial_mean.sel(warming_level=GWLS[0])["wspd10max"].median(dim="sim"),
    )

hot_day_freq    = float((_tmax_bl > tmax_threshold_f).mean())
warm_night_freq = float((_tmin_bl > tmin_threshold_f).mean())
calm_day_freq   = float((_wind_bl  < wind_ms_threshold).mean())

print(f"Thresholds:")
print(f"  tmax > {tmax_threshold_f}°F  → {hot_day_freq:.1%} of days (spatial median, GWL {GWLS[0]})")
print(f"  tmin > {tmin_threshold_f}°F  → {warm_night_freq:.1%} of days (spatial median, GWL {GWLS[0]})")
print(f"  wind < {wind_ms_threshold} m/s  → {calm_day_freq:.1%} of days (Solano, GWL {GWLS[0]})")

## Compute Compound Events for All GWLs

Loops over all 10 GWLs, computes per-sim compound event rates and sub-condition rates,
applies the sim-dropout corrections for incomplete simulations, then takes the median.

In [ ]:
def count_compound_events(tmax_da, tmin_da, wind_solano_da,
                           tmax_thresh_f, tmin_thresh_f, wind_ms_thresh,
                           n_years=30.0):
    """
    Compute annual compound event rate per grid cell, plus sub-condition rates.

    A compound event day satisfies all three conditions simultaneously:
      1. tmax > tmax_thresh_f  (hot day)
      2. tmin > tmin_thresh_f  (warm night)
      3. spatially-averaged Solano wind < wind_ms_thresh  (no-wind day)

    The wind condition is a scalar per day (already spatially averaged), so it
    applies uniformly to every SMUD grid cell. NaN wind values are treated as
    False (calm condition not met) via fillna.

    Parameters
    ----------
    tmax_da : xr.DataArray
        Daily max temperature (°F), dims (sim, time_delta, y, x).
    tmin_da : xr.DataArray
        Daily min temperature (°F), dims (sim, time_delta, y, x).
    wind_solano_da : xr.DataArray
        Spatially averaged daily max wind speed at Solano (m/s), dims (sim, time_delta).
    tmax_thresh_f : float
        Hot-day threshold in °F.
    tmin_thresh_f : float
        Warm-night threshold in °F.
    wind_ms_thresh : float
        Calm-wind threshold in m/s; days below this count as no-wind.
    n_years : float
        Length of sample window in years; used to convert day counts to annual rates.

    Returns
    -------
    dict of lazy xr.DataArrays:
        rate             : (sim, y, x)  compound events / yr per grid cell
        hot_day_rate     : (sim, y, x)  hot days / yr per grid cell
        warm_night_rate  : (sim, y, x)  warm nights / yr per grid cell
        calm_solano_rate : (sim,)       calm wind days / yr at Solano (no spatial dim)
        compound_count   : (sim, time_delta) grid cells with compound event per day
    """
    hot_day     = tmax_da > tmax_thresh_f
    warm_night  = tmin_da > tmin_thresh_f
    # fillna(False) so missing wind values don't propagate compound True flags
    calm_solano = (wind_solano_da < wind_ms_thresh).fillna(False)

    compound = hot_day & warm_night & calm_solano

    return {
        "rate":             compound.sum(dim="time_delta") / n_years,
        "hot_day_rate":     hot_day.sum(dim="time_delta") / n_years,
        "warm_night_rate":  warm_night.sum(dim="time_delta") / n_years,
        "calm_solano_rate": calm_solano.sum(dim="time_delta") / n_years,
        "compound_count":   compound.sum(dim=["y", "x"]),
    }


def drop_and_median(wl, da):
    """
    Drop simulations that have incomplete warming-level windows, then take the median.

    EC-Earth3-Veg and MPI-ESM1-2-HR do not reach GWL 0.75°C or 1.0°C within the
    SSP3-7.0 projection period, so they are excluded at those warming levels to
    avoid biasing the median with truncated 30-year windows.

    Parameters
    ----------
    wl : float
        Warming level being processed.
    da : xr.DataArray
        Array with a 'sim' dimension.

    Returns
    -------
    xr.DataArray
        Median across the remaining (complete) simulations.
    """
    sims_to_drop = {
        0.75: ["WRF_UCLA_EC-Earth3-Veg_ssp370_day_d03_r1i1p1f1",
               "WRF_UCLA_MPI-ESM1-2-HR_ssp370_day_d03_r3i1p1f1"],
        1.0:  ["WRF_UCLA_EC-Earth3-Veg_ssp370_day_d03_r1i1p1f1"],
    }
    if wl in sims_to_drop:
        da = da.drop_sel(sim=sims_to_drop[wl])
    return da.median(dim="sim")

In [ ]:
# Build the full lazy computation graph across all GWLs before triggering any compute.
# Deferring compute allows dask to optimise the task graph in a single batch call.
all_lazy = {}
for gwl in GWLS:
    all_lazy[gwl] = count_compound_events(
        t2max_data.sel(warming_level=gwl)["t2max"],
        t2min_data.sel(warming_level=gwl)["t2min"],
        wind_data_spatial_mean.sel(warming_level=gwl)["wspd10max"],
        tmax_threshold_f, tmin_threshold_f, wind_ms_threshold,
        n_years=n_years,
    )

# Flatten the nested {gwl: {metric: array}} structure into a single list so
# dask.compute can materialise everything in one pass.
metric_keys = list(next(iter(all_lazy.values())).keys())
lazy_vals = [all_lazy[gwl][k] for gwl in GWLS for k in metric_keys]

print(f"Computing {len(lazy_vals)} arrays across {len(GWLS)} GWLs...")
with ProgressBar():
    computed_flat = dask.compute(*lazy_vals)

# Unpack the flat results list back into per-GWL, per-metric dictionaries.
# Each block of n_metrics consecutive results corresponds to one GWL.
n_metrics = len(metric_keys)
events_by_gwl      = {}
hot_day_by_gwl     = {}
warm_night_by_gwl  = {}
calm_solano_by_gwl = {}

for i, gwl in enumerate(GWLS):
    result = dict(zip(metric_keys, computed_flat[i * n_metrics:(i + 1) * n_metrics]))

    # Apply sim-dropout correction and spatial mask, then add warming_level coord for concat
    events_by_gwl[gwl]      = drop_and_median(gwl, result["rate"]).where(smud_mask).expand_dims({"warming_level": [gwl]})
    hot_day_by_gwl[gwl]     = drop_and_median(gwl, result["hot_day_rate"]).where(smud_mask).expand_dims({"warming_level": [gwl]})
    warm_night_by_gwl[gwl]  = drop_and_median(gwl, result["warm_night_rate"]).where(smud_mask).expand_dims({"warming_level": [gwl]})
    calm_solano_by_gwl[gwl] = drop_and_median(gwl, result["calm_solano_rate"]).expand_dims({"warming_level": [gwl]})

    print(f"GWL {gwl}°C  compound events/yr: mean={float(events_by_gwl[gwl].mean()):.4f}, max={float(events_by_gwl[gwl].max()):.4f}")

## Save Outputs

Exports 4 netCDF files, each concatenated across all 10 GWLs:
- `heatwave_no_wind.nc` — compound event rate (events/yr per grid cell)
- `hot_day_rate.nc` — hot day frequency (days/yr per grid cell)
- `warm_night_rate.nc` — warm night frequency (days/yr per grid cell)
- `calm_solano_rate.nc` — calm wind days at Solano per year (scalar per GWL)

In [ ]:
# Concatenate across all 10 GWLs along a new warming_level dimension before exporting
export(xr.concat(list(events_by_gwl.values()),      dim="warming_level"), "data/heatwave_no_wind")
export(xr.concat(list(hot_day_by_gwl.values()),     dim="warming_level"), "data/hot_day_rate")
export(xr.concat(list(warm_night_by_gwl.values()),  dim="warming_level"), "data/warm_night_rate")
export(xr.concat(list(calm_solano_by_gwl.values()), dim="warming_level"), "data/calm_solano_rate")

Now that we've computed the compound event frequency across the SMUD service territory for all 10 GWLs, we can move on below to visualizing our results.

---

# Figures

In [ ]:
# Read back saved results for figure generation
compound_all    = xr.open_dataarray("data/heatwave_no_wind.nc")
hot_day_all     = xr.open_dataarray("data/hot_day_rate.nc")
warm_night_all  = xr.open_dataarray("data/warm_night_rate.nc")
calm_solano_all = xr.open_dataarray("data/calm_solano_rate.nc")

# Adding SMUD boundary as a GeoDataFrame for plotting.
smud_boundary = DataCatalog().boundaries._ca_utilities.query("Utility == 'Sacramento Municipal Utility District'")
SOLANO_WIND_FARM_CACHE = "data/solano_windfarm_area.zip"
solano_boundary = gpd.read_file(SOLANO_WIND_FARM_CACHE).to_crs('EPSG:4326')

maps_by_gwl        = {gwl: compound_all.sel(warming_level=gwl)    for gwl in GWLS}
hot_day_fig_gwl    = {gwl: hot_day_all.sel(warming_level=gwl)     for gwl in GWLS}
warm_night_fig_gwl = {gwl: warm_night_all.sel(warming_level=gwl)  for gwl in GWLS}
calm_solano_vals   = {gwl: float(calm_solano_all.sel(warming_level=gwl)) for gwl in GWLS}

years = get_gwl_year_map(GWLS)

In [ ]:
# Figure 1: SMUD service territory + Solano wind farm overview map
plot_smud_solano_grid_map(
    smud_boundary=smud_boundary,
    solano_gdf=solano_boundary,
    smud_grid=smud_mask,
    solano_grid=solano_mask,
    out_path="figures/smud_solano_grid_map.png",
)

In [ ]:
# Figure 2: Hot day and warm night sub-condition frequency maps (two saved PNGs)
plot_subcondition_frequency_maps(
    hot_day_fig_gwl,
    warm_night_fig_gwl,
    tmax_threshold=tmax_threshold_f,
    tmin_threshold=tmin_threshold_f,
    years=years,
    boundary=smud_boundary,
    out_dir="figures",
)

In [ ]:
# Figure 3: Solano wind farm calm days bar chart
plot_gwl_bar_chart(
    bars_by_label={f"{g}°C\n{years[g]}": calm_solano_vals[g] for g in GWLS},
    title="Solano Wind Farm — Low-Wind Days per Year by GWL",
    ylabel=f"Calm days / year (wind < {wind_ms_threshold} m/s)",
    out_path="figures/solano_calm_days_bar.png",
    ymin=70,
    ymax=80,
    figsize=(10, 5),
)

In [ ]:
# Figure 4: Compound event frequency maps across all 10 GWLs
plot_gwl_maps(
    maps_by_gwl,
    title="Hot Day + Warm Night + No-Wind Compound Event — Average Events per Grid Cell per Year",
    colorbar_label="Avg compound events / year",
    cmap="YlOrRd",
    out_path="figures/compound_frequency_maps.png",
    years=years,
    boundary=smud_boundary,
    marker_lon=SAC_LON,
    marker_lat=SAC_LAT,
)